Идея данного блока - соединить данные из таргетдата в один файл(образцы во время изменения и их характеристики по сути в одну таблицу)

In [28]:
import pandas as pd
# подключаем лист с названием Результаты в переменную метадата (в нем хранятся характеристики каждого образца)
metadata_df = pd.read_excel("data/TargetData.xlsx", sheet_name="Результаты")
# выводим все колонки до того как часть поменяем для объединения
print("Колонки до переименования:", metadata_df.columns.tolist())

Колонки до переименования: ['Unnamed: 0', 'Раствор полимера,%', 'Модификация волокна', 'Eмод, Гпа', 'Fmax, Н', 'sM, МПа', 'dL при Fмакс %', 'Длина, мм', 'Масса, мг', 'Unnamed: 9', 'Содержание волокна, %']


In [29]:

# переназываем первый столбец(колонку) на семплайди и меняем срзу,без создания копии
metadata_df.rename(columns={metadata_df.columns[0]: 'sample_id'}, inplace=True)
# Смотрим колонку с новым названием
print("Колонки после переименования:", metadata_df.columns.tolist())

Колонки после переименования: ['sample_id', 'Раствор полимера,%', 'Модификация волокна', 'Eмод, Гпа', 'Fmax, Н', 'sM, МПа', 'dL при Fмакс %', 'Длина, мм', 'Масса, мг', 'Unnamed: 9', 'Содержание волокна, %']


In [ ]:
# загружаем все листы сразу а после создаем новый словарь в котором будут только образцы без листа с результатом
all_sheets = pd.read_excel("data/TargetData.xlsx", sheet_name=None)
curve_sheets = {name: df for name, df in all_sheets.items() if name != 'Результаты'}
print(curve_sheets) # { "Образец1": [таблица кривой], "Образец2": [таблица кривой] ... }

{'Образец1.2':       Образец1.2        Образец1.2.1
0     Деформация  Стандартное усилие
1              %                 MPa
2              0           32.266912
3       0.000061           32.256458
4        0.00007           32.232026
...          ...                 ...
1507    2.377166             0.53416
1508    2.377021            0.471622
1509    2.377021            0.471622
1510    2.376767            0.464549
1511    2.376579            0.468017

[1512 rows x 2 columns], 'Образец1.3':       Образец1.3        Образец1.3.1
0     Деформация  Стандартное усилие
1              %                 MPa
2              0           32.141104
3        0.00003           32.144582
4       0.000173           32.081831
...          ...                 ...
1445    1.493738         2292.069672
1446    1.493735         2292.055664
1447    1.493735         2292.055664
1448    1.493731         2291.979309
1449    1.493721          2291.95459

[1450 rows x 2 columns], 'Образец1.4':       Образец1.4 

In [ ]:
# новый список для склеивания кривой и метаданных бок о бок
processed_dfs = []
# цикл где каждому текущему листу и таблице внутри него мы переработываем
for name, df in curve_sheets.items():
    curve = pd.read_excel("data/TargetData.xlsx", sheet_name=name, skiprows=3, header = None, names=["Деформация", "Напряжение"]) # загружаем текущий лист(а не следующий к примеру), игнорим первые 3 строки,говорим что заголвков нет и все данные и даем новые названия колонкам
    curve["sample_id"] = name # добавляем новую колонку с айди где написано какой образец(3 колонка)
    merged = pd.merge(curve, metadata_df, on="sample_id", how="left") # соединяем таблицы изменениями и методанными в один датафрпейм,объеднияем по колонке с айди и объединяем так(беря все строки из левой таблицы и приклеиваем им строки из правой)
    processed_dfs.append(merged) # добавляем этот новый датафрейм в список / СПИСОК: [Таблица1, Таблица2, Таблица3...]

In [ ]:
full_df = pd.concat(processed_dfs, ignore_index=True) # склеиваем таблицы друг с другом в одну большую
full_df.to_csv("data/PreparedData.csv", index=False, sep=";") # сохраняем в csv файл с разделением через ;
print(f"Готово! Строк: {len(full_df)}, Образцов: {full_df['sample_id'].nunique()}") # смотрим сколько у нас строк и уникальных образцов
print(full_df) # просто выводим финальную версию для примера(можно ее смотреть в отдельном файле или внутри визуал студио код скачав расширение

Готово! Строк: 163617, Образцов: 108
        Деформация  Напряжение    sample_id  Раствор полимера,%  \
0         0.000000   32.266912   Образец1.2                  20   
1         0.000061   32.256458   Образец1.2                  20   
2         0.000070   32.232026   Образец1.2                  20   
3         0.000073   32.246051   Образец1.2                  20   
4         0.000087   32.294809   Образец1.2                  20   
...            ...         ...          ...                 ...   
163612    1.935365    2.283912  Образец3.42                  50   
163613    1.935365    2.283912  Образец3.42                  50   
163614    1.936362    2.259605  Образец3.42                  50   
163615    1.938553    2.245597  Образец3.42                  50   
163616    1.938553    2.245597  Образец3.42                  50   

       Модификация волокна   Eмод, Гпа      Fmax, Н      sM, МПа  \
0                      нет  207.999590  1517.935303  3415.354431   
1                     